### Validacion de resultados del Paper

In [58]:
# Configuración de Hiperparámetros para modelos Linear, NLinear y DLinear
class Args:
    # Arquitectura y datos
    seq_len = 96        # Longitud de la ventana de entrada (lookback)
    label_len = 48      # Longitud de la etiqueta (usado en modelos con decoder)
    pred_len = 96       # Longitud de la ventana de predicción (horizonte)
    enc_in = 7          # Número de variables de entrada (features) - NOTA: Para univariante, este será 1.

    # Hiperparámetros de entrenamiento
    batch_size = 32     # Tamaño del lote
    learning_rate = 0.001
    num_epochs = 10

    # Hiperparámetro específico para DLinear
    # Determina el tamaño de la ventana para la descomposición de tendencia (moving average)
    moving_avg_kernel = 25

    # General
    dropout = 0.05      # Regularización
    use_gpu = True      # Uso de aceleración por hardware

# Instancia de configuración
args = Args()

print(f"Configuración cargada:")
print(f"- Ventana de entrada: {args.seq_len}")
print(f"- Horizonte de predicción: {args.pred_len}")
print(f"- Kernel para DLinear: {args.moving_avg_kernel}")
print(f"- Número de épocas: {args.num_epochs}")
print(f"- Tasa de aprendizaje: {args.learning_rate}")
print(f"- Tamaño del lote: {args.batch_size}")

Configuración cargada:
- Ventana de entrada: 96
- Horizonte de predicción: 96
- Kernel para DLinear: 25
- Número de épocas: 10
- Tasa de aprendizaje: 0.001
- Tamaño del lote: 32


In [60]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import MinMaxScaler
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# Importar la instancia de args definida en la celda anterior
# from <previous_cell> import args # No es necesario si se ejecuta en el mismo entorno

# Lista de URLs de los datasets ETT
dataset_urls = {
    "ETTh1": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv",
    "ETTh2": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh2.csv",
    "ETTm1": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm1.csv",
    "ETTm2": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm2.csv"
}

# Definición de la clase TimeSeriesDataset (movida aquí para que esté disponible en el bucle)
class TimeSeriesDataset(Dataset):
    def __init__(self, data, seq_len, pred_len):
        self.data = data
        self.seq_len = seq_len
        self.pred_len = pred_len

    def __len__(self):
        return len(self.data) - self.seq_len - self.pred_len + 1

    def __getitem__(self, idx):
        seq_x = self.data[idx : idx + self.seq_len]
        seq_y = self.data[idx + self.seq_len : idx + self.seq_len + self.pred_len]
        return seq_x, seq_y

# Definición de la clase Configs (movida aquí)
class Configs:
    def __init__(self, seq_len, pred_len, enc_in):
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.enc_in = enc_in


# Modelos Linear, NLinear, DLinear (movidos aquí para que estén disponibles en el bucle)
class Linear(nn.Module):
    def __init__(self, configs):
        super(Linear, self).__init__()
        self.linear = nn.Linear(configs.seq_len, configs.pred_len)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.linear(x)
        return x.permute(0, 2, 1)

class NLinear(nn.Module):
    def __init__(self, configs):
        super(NLinear, self).__init__()
        self.linear = nn.Linear(configs.seq_len, configs.pred_len)

    def forward(self, x):
        seq_last = x[:, -1:, :].detach()
        x = x - seq_last
        x = self.linear(x.permute(0, 2, 1)).permute(0, 2, 1)
        return x + seq_last

class moving_avg(nn.Module):
    def __init__(self, kernel_size, stride):
        super(moving_avg, self).__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=stride, padding=0)

    def forward(self, x):
        front = x[:, 0:1, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        end = x[:, -1:, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        x = torch.cat([front, x, end], dim=1)
        x = self.avg(x.permute(0, 2, 1))
        return x.permute(0, 2, 1)

class DLinear(nn.Module):
    def __init__(self, configs, kernel_size):
        super(DLinear, self).__init__()
        self.decomposition = moving_avg(kernel_size, stride=1)
        self.Linear_Seasonal = nn.Linear(configs.seq_len, configs.pred_len)
        self.Linear_Trend = nn.Linear(configs.seq_len, configs.pred_len)

    def forward(self, x):
        seasonal_init, trend_init = self.decomposition(x), x - self.decomposition(x)
        seasonal_output = self.Linear_Seasonal(seasonal_init.permute(0, 2, 1)).permute(0, 2, 1)
        trend_output = self.Linear_Trend(trend_init.permute(0, 2, 1)).permute(0, 2, 1)
        return seasonal_output + trend_output

# Función para entrenar un modelo
def train_model(model, dataloader, optimizer, criterion, epochs):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch_x, batch_y in dataloader:
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        # print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(dataloader):.4f}") # Comentado para no saturar la salida

# Función para evaluar un modelo
def evaluate_model(model, dataloader, scaler):
    model.eval()
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            outputs = model(batch_x)
            all_preds.append(outputs.cpu().numpy())
            all_targets.append(batch_y.cpu().numpy())

    preds = np.concatenate(all_preds, axis=0)
    targets = np.concatenate(all_targets, axis=0)

    # Inverse transform predictions and targets to original scale
    preds_original_scale = scaler.inverse_transform(preds.reshape(-1, 1))
    targets_original_scale = scaler.inverse_transform(targets.reshape(-1, 1))

    avg_mse = mean_squared_error(targets_original_scale, preds_original_scale)
    avg_mae = mean_absolute_error(targets_original_scale, preds_original_scale)
    return avg_mse, avg_mae


results = {}

# Usar los argumentos definidos globalmente
seq_len = args.seq_len
pred_len = args.pred_len
batch_size = args.batch_size
learning_rate = args.learning_rate
num_epochs = args.num_epochs
moving_avg_kernel = args.moving_avg_kernel

for dataset_name, url in dataset_urls.items():
    print(f"\n--- Procesando dataset: {dataset_name} ---")

    # 1. Cargar los datos
    df = pd.read_csv(url)

    # Para univariante forecasting, seleccionamos solo la columna 'OT' (Oil Temperature)
    data_values = df[['OT']].values
    data_tensor = torch.tensor(data_values, dtype=torch.float32)

    # Normalización de los datos
    scaler = MinMaxScaler()
    data_normalized = scaler.fit_transform(data_values)
    data_tensor_normalized = torch.tensor(data_normalized, dtype=torch.float32)

    # División en conjuntos de entrenamiento y prueba (80%/20%)
    train_size = int(0.8 * len(data_tensor_normalized))
    train_data = data_tensor_normalized[:train_size]
    test_data = data_tensor_normalized[train_size:]

    # Crear DataSets
    train_dataset = TimeSeriesDataset(train_data, seq_len, pred_len)
    test_dataset = TimeSeriesDataset(test_data, seq_len, pred_len)

    # Crear DataLoaders para entrenamiento y prueba
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Configs para este dataset (enc_in es 1 para univariante)
    configs = Configs(seq_len, pred_len, enc_in=1)

    current_dataset_results = {}

    # Entrenamiento y evaluación de Linear
    model_linear = Linear(configs)
    optimizer_linear = optim.Adam(model_linear.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()
    train_model(model_linear, train_dataloader, optimizer_linear, criterion, epochs=num_epochs)
    mse_linear, mae_linear = evaluate_model(model_linear, test_dataloader, scaler)
    current_dataset_results['Linear'] = {'MSE': mse_linear, 'MAE': mae_linear}
    print(f"Linear  - MSE: {mse_linear:.4f}, MAE: {mae_linear:.4f}")

    # Entrenamiento y evaluación de NLinear
    model_nlinear = NLinear(configs)
    optimizer_nlinear = optim.Adam(model_nlinear.parameters(), lr=learning_rate)
    train_model(model_nlinear, train_dataloader, optimizer_nlinear, criterion, epochs=num_epochs)
    mse_nlinear, mae_nlinear = evaluate_model(model_nlinear, test_dataloader, scaler)
    current_dataset_results['NLinear'] = {'MSE': mse_nlinear, 'MAE': mae_nlinear}
    print(f"NLinear - MSE: {mse_nlinear:.4f}, MAE: {mae_nlinear:.4f}")

    # Entrenamiento y evaluación de DLinear
    model_dlinear = DLinear(configs, kernel_size=moving_avg_kernel)
    optimizer_dlinear = optim.Adam(model_dlinear.parameters(), lr=learning_rate)
    train_model(model_dlinear, train_dataloader, optimizer_dlinear, criterion, epochs=num_epochs)
    mse_dlinear, mae_dlinear = evaluate_model(model_dlinear, test_dataloader, scaler)
    current_dataset_results['DLinear'] = {'MSE': mse_dlinear, 'MAE': mae_dlinear}
    print(f"DLinear - MSE: {mse_dlinear:.4f}, MAE: {mae_dlinear:.4f}")

    results[dataset_name] = current_dataset_results

print("\n--- Resumen de Todos los Resultados ---")
for dataset_name, dataset_results in results.items():
    print(f"\nDataset: {dataset_name}")
    for model_name, metrics in dataset_results.items():
        print(f"  {model_name:<7} - MSE: {metrics['MSE']:.4f}, MAE: {metrics['MAE']:.4f}")


--- Procesando dataset: ETTh1 ---
Linear  - MSE: 7.8467, MAE: 2.1365
NLinear - MSE: 7.5853, MAE: 2.1137
DLinear - MSE: 7.3403, MAE: 2.0961

--- Procesando dataset: ETTh2 ---
Linear  - MSE: 36.3659, MAE: 4.5750
NLinear - MSE: 35.5503, MAE: 4.5384
DLinear - MSE: 35.8467, MAE: 4.5821

--- Procesando dataset: ETTm1 ---
Linear  - MSE: 3.4024, MAE: 1.3808
NLinear - MSE: 3.2207, MAE: 1.2899
DLinear - MSE: 4.0106, MAE: 1.5755

--- Procesando dataset: ETTm2 ---
Linear  - MSE: 16.8379, MAE: 2.8386
NLinear - MSE: 16.3988, MAE: 2.8278
DLinear - MSE: 15.8670, MAE: 2.7704

--- Resumen de Todos los Resultados ---

Dataset: ETTh1
  Linear  - MSE: 7.8467, MAE: 2.1365
  NLinear - MSE: 7.5853, MAE: 2.1137
  DLinear - MSE: 7.3403, MAE: 2.0961

Dataset: ETTh2
  Linear  - MSE: 36.3659, MAE: 4.5750
  NLinear - MSE: 35.5503, MAE: 4.5384
  DLinear - MSE: 35.8467, MAE: 4.5821

Dataset: ETTm1
  Linear  - MSE: 3.4024, MAE: 1.3808
  NLinear - MSE: 3.2207, MAE: 1.2899
  DLinear - MSE: 4.0106, MAE: 1.5755

Dataset:

## Consideraciones sobre la Reproducción de Resultados de Papers

# https://arxiv.org/abs/2205.13504

Es importante destacar que **obtener una replicación exacta de los resultados presentados del paper científico han sido un desafío**, se ha considerado la busqueda dentro y fuera de la documentacion entregada, asi como la opcion que tenemos en incurrir en la formacion de lineamientos de investigacion proporcionada por la IA.

Para acercarse a los resultados del paper, sería necesario:

*   **Hiperparámetros:** Realizar una búsqueda exhaustiva de hiperparámetros (`seq_len`, `pred_len`, `batch_size`, `learning_rate`, `num_epochs`, `moving_avg_kernel`, `dropout`, `set_seed()`, etc.) utilizando técnicas como Grid Search o Random Search.
*   **DataSet:** Obtencion de los dataset, con los que eleboraron el documento.

La implementación actual es una base sólida que demuestra la funcionalidad de los modelos, pero no busca replicar los resultados de un paper optimizado al milímetro sin los recursos y el tiempo asociados a la investigación en ese nivel.